# Titanic: feature engineering + soft-vote ensemble
Title/family/ticket/deck features, median age imputation by title and class, repeated 5x5 CV to pick models, soft-vote of the top 3. Writes `submission.csv`.

In [ ]:
import os, numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

import glob
hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
D = os.path.dirname(hits[0]) if hits else '.'
print('data dir:', D)
train = pd.read_csv(f'{D}/train.csv'); test = pd.read_csv(f'{D}/test.csv')
full = pd.concat([train, test], sort=False, ignore_index=True)

In [ ]:
def features(df):
    d = pd.DataFrame(index=df.index)
    title = df.Name.str.extract(r' ([A-Za-z]+)\.', expand=False).replace(
        {'Mlle':'Miss','Ms':'Miss','Mme':'Mrs','Lady':'Rare','Countess':'Rare','Dona':'Rare',
         'Sir':'Rare','Don':'Rare','Jonkheer':'Rare','Capt':'Rare','Col':'Rare','Major':'Rare','Dr':'Rare','Rev':'Rare'})
    d['Title'] = title
    d['Sex'] = (df.Sex == 'female').astype(int)
    d['Pclass'] = df.Pclass
    d['FamilySize'] = df.SibSp + df.Parch + 1
    d['IsAlone'] = (d.FamilySize == 1).astype(int)
    d['Deck'] = df.Cabin.str[0].fillna('U')
    d['HasCabin'] = df.Cabin.notna().astype(int)
    tk = df.Ticket.map(df.Ticket.value_counts())
    d['TicketGroup'] = tk
    d['Fare'] = df.Fare.fillna(df.groupby('Pclass').Fare.transform('median'))
    d['FarePP'] = d.Fare / tk
    d['Embarked'] = df.Embarked.fillna('S')
    d['Age'] = df.Age.fillna(df.groupby([title, df.Pclass]).Age.transform('median')).fillna(df.Age.median())
    d['IsChild'] = (d.Age < 13).astype(int)
    d['SurnameSize'] = df.Name.str.split(',').str[0].map(df.Name.str.split(',').str[0].value_counts())
    return pd.get_dummies(d, columns=['Title','Deck','Embarked'], dtype=int)

X_all = features(full)
n = len(train)
X, y, X_test = X_all.iloc[:n], train.Survived.values, X_all.iloc[n:]

In [ ]:
models = {
  'logreg': make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=2000)),
  'rf': RandomForestClassifier(n_estimators=500, min_samples_leaf=3, max_features='sqrt', random_state=0, n_jobs=-1),
  'gb': GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=0),
  'hgb': HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, l2_regularization=1.0, random_state=0),
}
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
scores = {k: cross_val_score(m, X, y, cv=cv, scoring='accuracy').mean() for k, m in models.items()}
for k, v in sorted(scores.items(), key=lambda kv: -kv[1]): print(f'{k:7s} CV acc {v:.4f}')
best = max(scores, key=scores.get)
print('best:', best)

# soft-vote of the top models is usually more stable than a single one
top = sorted(scores, key=scores.get, reverse=True)[:3]
for m in top: models[m].fit(X, y)
proba = np.mean([models[m].predict_proba(X_test)[:, 1] for m in top], axis=0)
sub = pd.DataFrame({'PassengerId': test.PassengerId, 'Survived': (proba >= 0.5).astype(int)})
sub.to_csv('submission.csv', index=False)
print('ensemble of', top, '| rows', len(sub), '| survival rate', sub.Survived.mean().round(3))